# <p style="text-align: center; color: red;"> Modelling and Control of Cement Emissions using ML</p> 

# <p style="text-align: left; color: yellow;"> Sheikh Junaid Fayaz -- M3RG Lab -- Indian Institute of Technology, Delhi</p> 
# <p style="text-align: left; color: yellow;"> Date : 06 March 2026</p> 

In [ ]:
import sys
sys.path.append('/media/m3rg2000/mounted/Junaid/.../Emissions/new_emission/Notebooks')
import importlib
import functions  # First import
importlib.reload(functions)  # Reloads the module to reflect changes
from functions import *  # Now import functions.py

In [2]:
import platform
import psutil
import cpuinfo
import GPUtil
import os
import sys
import pkg_resources
import numpy as np
import pandas as pd
import sklearn
import xgboost
import optuna
import skopt
from skopt import BayesSearchCV
import shap
import matplotlib
import seaborn as sns
import geopandas as gpd
from importlib.metadata import version
import shapely
import pynvml
from importlib.metadata import version
import importlib.metadata

In [6]:

print("### SYSTEM INFORMATION ###")
# OS & Python
print(f"OS: {platform.system()} {platform.release()} ({platform.version()})")
print(f"Python Version: {platform.python_version()}")

# CPU
cpu = cpuinfo.get_cpu_info()
print(f"CPU: {cpu['brand_raw']}")
print(f"Physical Cores: {psutil.cpu_count(logical=False)}")
print(f"Logical Cores: {psutil.cpu_count(logical=True)}")
print(f"CPU Frequency: {psutil.cpu_freq().max:.2f} MHz")

# RAM
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"Total RAM: {ram_gb:.2f} GB")

# GPU
print("\n### GPU INFORMATION ###")
gpus = GPUtil.getGPUs()
if gpus:
    for i, gpu in enumerate(gpus):
        print(f"GPU {i}: {gpu.name}")
        print(f"  Driver Version: {gpu.driver}")
        print(f"  Memory Total: {gpu.memoryTotal:.2f} MB")
        print(f"  Memory Free: {gpu.memoryFree:.2f} MB")
        print(f"  Memory Used: {gpu.memoryUsed:.2f} MB")
        print(f"  GPU Load: {gpu.load * 100:.1f}%")
else:
    print("No GPU detected.")

# Optional: System architecture
print(f"\nArchitecture: {platform.machine()} ({platform.architecture()[0]})")
print(f"Processor: {platform.processor()}")

### SYSTEM INFORMATION ###
OS: Linux 5.19.0-41-generic (#42~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 18 17:40:00 UTC 2)
Python Version: 3.9.12
CPU: Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
Physical Cores: 32
Logical Cores: 64
CPU Frequency: 3900.00 MHz
Total RAM: 62.54 GB

### GPU INFORMATION ###
GPU 0: NVIDIA RTX A2000 12GB
  Driver Version: 525.105.17
  Memory Total: 12282.00 MB
  Memory Free: 11454.00 MB
  Memory Used: 582.00 MB
  GPU Load: 0.0%

Architecture: x86_64 (64bit)
Processor: x86_64


### Summary of the above

Computational Environment. All experiments were conducted on a Linux machine running Ubuntu 22.04 with Python 3.9.12. The system was equipped with an Intel® Xeon® Gold 6226R CPU (32 physical cores, 64 threads) and 62.5 GB of RAM. GPU-accelerated models, such as XGBoost, were trained using an NVIDIA RTX A2000 12 GB GPU with driver version 525.105.17, utilizing the gpu_hist tree method.

In [9]:
print("\n--- Core Libraries ---")
print(f"os: built-in")  
print(f"time: built-in")  
print(f"random: built-in")  
print(f"subprocess: built-in")  
print(f"threading: built-in")  
print(f"resource: built-in")  
print(f"statistics: built-in") 
 
print("\n--- Data handling ---")
print(f"pandas: {pd.__version__}")  
print(f"numpy: {np.__version__}")  

print("\n--- Machine Learning Libraries ---")
print(f"scikit-learn: {sklearn.__version__}")  
print(f"xgboost: {xgboost.__version__}")  
print(f"optuna: {optuna.__version__}")  
print(f"RandomizedSearchCV (scikit-learn): {sklearn.__version__}")
print(f"BayesSearchCV (scikit-optimize): {skopt.__version__}")
print(f"scikit-optimize (skopt): {skopt.__version__}")  

print("\n--- Model Interpretation Libraries ---")
print(f'shap: {shap.__version__}')

print("\n--- Visualization Libraries ---")
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn: {sns.__version__}')

print("\n--- Geospatial Libraries ---")
print(f'geopandas: {gpd.__version__}')
print(f"shapely: {shapely.__version__}")  

print("\n--- GPU Libraries ---")
print(f"pynvml: {version('nvidia-ml-py3')}")

print("\n--- Progress bar---")
print(f"tqdm: {importlib.metadata.version('tqdm')}")


--- Core Libraries ---
os: built-in
time: built-in
random: built-in
subprocess: built-in
threading: built-in
resource: built-in
statistics: built-in

--- Data handling ---
pandas: 1.4.2
numpy: 1.23.5

--- Machine Learning Libraries ---
scikit-learn: 1.2.2
xgboost: 1.7.4
optuna: 4.4.0
RandomizedSearchCV (scikit-learn): 1.2.2
BayesSearchCV (scikit-optimize): 0.10.2
scikit-optimize (skopt): 0.10.2

--- Model Interpretation Libraries ---
shap: 0.41.0

--- Visualization Libraries ---
matplotlib: 3.5.1
seaborn: 0.12.2

--- Geospatial Libraries ---
geopandas: 1.0.1
shapely: 2.0.1

--- GPU Libraries ---
pynvml: 7.352.0

--- Progress bar---
tqdm: 4.65.0


In [15]:
# Library info organized by category
libraries_info = [
    ("Core", "os", "built-in"),
    ("Core", "time", "built-in"),
    ("Core", "random", "built-in"),
    ("Core", "subprocess", "built-in"),
    ("Core", "threading", "built-in"),
    ("Core", "resource", "built-in"),
    ("Core", "statistics", "built-in"),

    ("Data Handling", "pandas", pd.__version__),
    ("Data Handling", "numpy", np.__version__),

    ("Machine Learning", "scikit-learn", sklearn.__version__),
    ("Machine Learning", "xgboost", xgboost.__version__),
    ("Machine Learning", "optuna", optuna.__version__),
    ("Machine Learning", "RandomizedSearchCV", sklearn.__version__),
    ("Machine Learning", "BayesSearchCV", skopt.__version__),
    ("Machine Learning", "scikit-optimize", skopt.__version__),

    ("Model Interpretation", "shap", shap.__version__),

    ("Visualization", "matplotlib", matplotlib.__version__),
    ("Visualization", "seaborn", sns.__version__),

    ("Geospatial", "geopandas", gpd.__version__),
    ("Geospatial", "shapely", shapely.__version__),

    ("GPU", "pynvml", version("nvidia-ml-py3")),

    ("Progress bar", "tqdm", version("tqdm")),
]

# Create DataFrame
df = pd.DataFrame(libraries_info, columns=["library_type", "library_name", "library_version"])
df
df.to_excel('/home/m3rg2000/Junaid_temporary/saved_data/general/library_config.xlsx', index = False)

,library_type,library_name,library_version
0,Core,os,built-in
1,Core,time,built-in
2,Core,random,built-in
3,Core,subprocess,built-in
4,Core,threading,built-in
5,Core,resource,built-in
6,Core,statistics,built-in
7,Data Handling,pandas,1.4.2
8,Data Handling,numpy,1.23.5
9,Machine Learning,scikit-learn,1.2.2


In [8]:
installed = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
installed

{'certifi': '2023.7.22',
 'pytz': '2022.7.1',
 'setuptools': '65.6.3',
 'pyaml': '25.7.0',
 'flatbuffers': '25.2.10',
 'pyzmq': '25.0.1',
 'pip': '23.0.1',
 'packaging': '23.0',
 'attrs': '22.2.0',
 'argon2-cffi': '21.3.0',
 'argon2-cffi-bindings': '21.2.0',
 'isoduration': '20.11.0',
 'pyarrow': '19.0.0',
 'lit': '16.0.0',
 'libclang': '16.0.0',
 'rich': '14.0.0',
 'nvidia-cublas-cu11': '11.10.3.66',
 'nvidia-cuda-cupti-cu11': '11.7.101',
 'nvidia-cuda-nvrtc-cu11': '11.7.99',
 'nvidia-cuda-runtime-cu11': '11.7.99',
 'nvidia-nvtx-cu11': '11.7.91',
 'nvidia-cusparse-cu11': '11.7.4.91',
 'nvidia-cusolver-cu11': '11.4.0.1',
 'nvidia-cufft-cu11': '10.9.0.58',
 'nvidia-curand-cu11': '10.2.10.91',
 'pillow': '9.4.0',
 'py-cpuinfo': '9.0.0',
 'ipython': '8.11.0',
 'nvidia-cudnn-cu11': '8.5.0.96',
 'ipywidgets': '8.0.4',
 'jupyter-client': '8.0.3',
 'nvidia-ml-py3': '7.352.0',
 'lab': '7.3',
 'nbconvert': '7.2.10',
 'ipykernel': '6.21.3',
 'colorlog': '6.9.0',
 'jupyter-console': '6.6.3',
 'no